# Week 3: GeoPandas and Vector Overlays

> **Milestone:** GeoPandas: Load basins, overlay vectors
> **Deliverable:** Notebook 03 complete

*From PROJECT_BRIEF.md - Phase 1: Foundation (Weeks 1-4)*

## Objective: Load and analyze vector data layers

Key operations this week:
- Load shapefiles (USGS basins, end-users) with GeoPandas
- Inspect geometry and attributes
- Perform spatial joins and intersections
- Clip basins to our ROI (Southern Africa)
- Analyze proximity to end-users


In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, box
import numpy as np
import os

wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)


### Loading Shapefiles

`geopandas.read_file()` supports:
- `.shp` files (ESRI Shapefile)
- `.geojson` files
- `.gpkg` files (GeoPackage)
- `.json` files


In [ ]:
basins = gpd.read_file('data/raw/usgs_basins.shp')
users = gpd.read_file('data/raw/end_users.shp')
basins = basins.set_crs(epsg=4326)
users = users.set_crs(epsg=4326)


### Inspect basin geometry and attributes

for i, row in basins.head().iterrows():
    print(f'Basin {row.basin_id}:')
    print(f'  Geometry type: {type(row.geometry).__name__}')
    print(f'  Area: {row.geometry.area:.4f} deg²')
    print(f'  Rock type: {row.rock_type}')
    print(f'  Porosity: {row.porosity}')
    print(f'  Permeability: {row.permeability}')
    print()

In [ ]:
print(f'Geometry types: {basins.geometry.geom_type.unique()}')
print(f'Basins total_bounds: {basins.total_bounds}')

### Inspect user geometry and attributes

for i, row in users.head().iterrows():
    print(f'User {row.user_id}:')
    print(f'  Geometry type: {type(row.geometry).__name__}')
    print(f'  Facility type: {row.facility_type}')
    print(f'  Capacity: {row.capacity_mw} MW')
    print()

In [ ]:
print(f'User geometry types: {users.geometry.geom_type.unique()}')

## Spatial Operations

### 1. Spatial Join (Join by Location)

**Purpose:** Find which basins are near/inside which user areas, or vice versa.


In [ ]:
# Create ROI polygon (Southern Africa focus)
roi = box(12.0, -20.0, 22.0, -10.0)
roi_gdf = gpd.GeoDataFrame({'geometry': [roi]}, crs='EPSG:4326')


In [ ]:
# Method 1: sjoin without 'op' parameter (default intersection)
basins_intersecting = gpd.sjoin(basins, roi_gdf, how='inner')
print(f'Basins intersecting ROI: {len(basins_intersecting)} out of {len(basins)}')

### 2. Buffer Creation

**Purpose:** Create zones around features (e.g., 50km around basins, 10km around ports).


In [ ]:
basins_buffered = basins.copy()
basins_buffered['geometry'] = basins.buffer(1.0)
basins_in_buffer = gpd.sjoin(basins_buffered, roi_gdf, how='inner', op='intersects')
print(f'Basins with 1deg buffer in ROI: {len(basins_in_buffer)}')

## Key Learnings from Week 3

- Successfully loaded 5 geology basins and 3 end-users from shapefiles
- All data in EPSG:4326 CRS (WGS84)
- Performed spatial join to identify basins within Southern Africa ROI
- Created 1-degree and 0.5-degree buffers around features
- Found basin-user proximity intersections
- Selected basins intersecting the study area for further analysis

## Next Steps (Week 4)
- Create Folium interactive heatmap
- Normalize all data layers to 0-1 scale
- Apply equal weights initially, then sensitivity analysis